In [1]:
import importlib.util
import json
import os
import sys
from itertools import product
from pathlib import Path

sys.dont_write_bytecode = True

# ---------------------------------------------------------------------------
# Load generate_mul_mat_suite / generate_soft_max_suite modules
# ---------------------------------------------------------------------------
_SUITE_SCRIPT = Path("scripts/generate_mul_mat_suite.py")
if not _SUITE_SCRIPT.exists():
    raise FileNotFoundError(f"Suite script not found: {_SUITE_SCRIPT.resolve()}")
_spec = importlib.util.spec_from_file_location("generate_mul_mat_suite", _SUITE_SCRIPT)
_suite_mod = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _suite_mod
_spec.loader.exec_module(_suite_mod)

_SOFTMAX_SUITE_SCRIPT = Path("scripts/generate_soft_max_suite.py")
if not _SOFTMAX_SUITE_SCRIPT.exists():
    raise FileNotFoundError(f"Suite script not found: {_SOFTMAX_SUITE_SCRIPT.resolve()}")
_softmax_spec = importlib.util.spec_from_file_location("generate_soft_max_suite", _SOFTMAX_SUITE_SCRIPT)
_softmax_suite_mod = importlib.util.module_from_spec(_softmax_spec)
sys.modules[_softmax_spec.name] = _softmax_suite_mod
_softmax_spec.loader.exec_module(_softmax_suite_mod)


# ---------------------------------------------------------------------------
# Shell config helpers
# ---------------------------------------------------------------------------

def declare_array(f, name, list):
    f.write("declare -a {}_array=(\n".format(name))
    for i in list:
        f.write('  "{}" \n'.format(i))
    f.write(")\n")


def generate_synth_bench(
    synth_params,
    output_path="configs/synth_tests.txt",
):
    """Generate a MUL_MAT test file from synth_params.

    Returns (test_file_path_str, list_of_test_names).
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    type_map = _suite_mod.load_type_map_from_header(_suite_mod.DEFAULT_GGML_H)

    lines = []
    names = []
    for idx, (type_a, type_b, m, n, k) in enumerate(
        product(
            synth_params["type_a"],
            synth_params["type_b"],
            synth_params["m"],
            synth_params["n"],
            synth_params["k"],
        ),
        start=1,
    ):
        type_a_id = _suite_mod.parse_type(str(type_a), type_map)
        type_b_id = _suite_mod.parse_type(str(type_b), type_map)
        name = f"{type_a}_{m}M_{n}N_{k}K".replace("_K", "").replace("GGML_TYPE_", "")
        line = _suite_mod.build_mul_mat_line(
            type_a=type_a_id,
            type_b=type_b_id,
            m=int(m),
            n=int(n),
            k=int(k),
            name=name,
        )
        lines.append(line)
        names.append(name)

    # output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Generated {len(lines)} synth test lines → {output_path}")
    return str(output_path), names, lines


def generate_soft_max_bench(
    synth_softmax_params,
    output_path="configs/synth_softmax_tests.txt",
):
    """Generate a SOFT_MAX test file from synth_softmax_params (same shape as
    benchmark/configs/synth_configs/synth_softmax_params.json: ne0/ne1/ne2/ne3,
    mask_variants, sinks, scale, max_bias — see generate_soft_max_suite.py).

    Returns (test_file_path_str, list_of_test_names, list_of_test_lines).
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    type_map = _softmax_suite_mod.load_type_map_from_header(_softmax_suite_mod.DEFAULT_GGML_H)
    f32_id = _softmax_suite_mod.parse_type("f32", type_map)

    lines = []
    names = []
    idx = 0
    for ne0, ne1, ne2, ne3, mask_variant, sinks, scale, max_bias in product(
        synth_softmax_params["ne0"],
        synth_softmax_params["ne1"],
        synth_softmax_params["ne2"],
        synth_softmax_params["ne3"],
        synth_softmax_params["mask_variants"],
        synth_softmax_params["sinks"],
        synth_softmax_params["scale"],
        synth_softmax_params["max_bias"],
    ):
        idx += 1
        mask = bool(mask_variant.get("mask", False))
        mask_type_id = None
        mask_tag = "noMask"
        if mask:
            mask_type_id = _softmax_suite_mod.parse_type(str(mask_variant["mask_type"]), type_map)
            mask_tag = f"Mask{str(mask_variant['mask_type']).upper()}"

        name = (
            f"SOFT_MAX_{ne0}x{ne1}x{ne2}x{ne3}_{mask_tag}"
            f"{'_Sinks' if sinks else ''}_s{scale}_b{max_bias}"
        )

        try:
            line = _softmax_suite_mod.build_soft_max_line(
                f32_type_id=f32_id,
                ne=(int(ne0), int(ne1), int(ne2), int(ne3)),
                mask=mask,
                mask_type_id=mask_type_id,
                sinks=bool(sinks),
                scale=float(scale),
                max_bias=float(max_bias),
                name=name,
            )
        except ValueError as exc:
            print(f"Skipping invalid combo {name}: {exc}")
            continue
        lines.append(line)
        names.append(name)

    print(f"Generated {len(lines)} synth softmax test lines → {output_path}")
    return str(output_path), names, lines


def generate_config(
    runtime_dict,
    models_dict,
    runtimes,
    models,
    output_path="configs/exp_configs.sh",
    synth_params=None,
    synth_output_path="configs/synth_tests.txt",
    synth_softmax_params=None,
    synth_softmax_output_path="configs/synth_softmax_tests.txt",
):
    bitstreams_list = [runtime_dict[runtime]["bitstreams"] for runtime in runtimes]
    binaries_list = [runtime_dict[runtime]["binaries"] for runtime in runtimes]
    acc_tags_list = [runtime_dict[runtime]["acc_tag"] for runtime in runtimes]
    bins_list = [runtime_dict[runtime]["bins"] for runtime in runtimes]
    accelerated_list = [runtime_dict[runtime]["accelerated"] for runtime in runtimes]
    models_list = [models_dict[model] for model in models]
    model_names_list = [model for model in models]
    bin_flags_list = [runtime_dict[runtime]["bin_flags"] for runtime in runtimes]

    synth_names = []
    synth_lines = []
    if synth_params is not None:
        _, mm_names, mm_lines = generate_synth_bench(synth_params, synth_output_path)
        synth_names += mm_names
        synth_lines += mm_lines
    if synth_softmax_params is not None:
        _, sm_names, sm_lines = generate_soft_max_bench(synth_softmax_params, synth_softmax_output_path)
        synth_names += sm_names
        synth_lines += sm_lines

    with open(output_path, "w+") as f:
        declare_array(f, "models", models_list)
        declare_array(f, "model_names", model_names_list)
        declare_array(f, "bitstreams", bitstreams_list)
        declare_array(f, "binaries", binaries_list)
        declare_array(f, "acc_tags", acc_tags_list)
        declare_array(f, "bins", bins_list)
        declare_array(f, "accelerated", accelerated_list)
        declare_array(f, "bin_flags", bin_flags_list)
        declare_array(f, "runtimes", runtimes)
        if synth_names:
            declare_array(f, "synth_test_names", synth_names)
            declare_array(f, "synth_test_lines", synth_lines)


def load_exp_config(config_path):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Experiment config file not found: {config_path}")
    with open(config_path, "r") as jf:
        exp_config = json.load(jf)
    return exp_config["models"], exp_config["runtimes"]


# ---------------------------------------------------------------------------
# Load config dictionaries
# ---------------------------------------------------------------------------

# Runtimes (Accelerators)
with open("configs/runtimes/runtime_dict_og.json", "r") as jf:
    runtime_dict = json.load(jf)

with open("configs/runtimes/runtime_dict_1.json", "r") as jf:
    runtime_dict_1 = json.load(jf)

with open("configs/runtimes/runtime_dict_2.json", "r") as jf:
    runtime_dict_2 = json.load(jf)

with open("configs/runtimes/runtime_dict_3.json", "r") as jf:
    runtime_dict_3 = json.load(jf)

with open("configs/runtimes/runtime_dict_6.json", "r") as jf:
    runtime_dict_6 = json.load(jf)

with open("configs/runtimes/runtime_dict_sllm.json", "r") as jf:
    runtime_dict_sllm = json.load(jf)

runtime_dict.update(runtime_dict_1)
runtime_dict.update(runtime_dict_2)
runtime_dict.update(runtime_dict_3)
runtime_dict.update(runtime_dict_6)
runtime_dict.update(runtime_dict_sllm)

# Models
with open("configs/models/models_dict.json", "r") as jf:
    models_dict = json.load(jf)

# Synthetic benchmark params (default)
with open("configs/synth_configs/synth_params.json", "r") as jf:
    synth_dict = json.load(jf)

# Synthetic SOFT_MAX benchmark params (default)
with open("configs/synth_configs/synth_softmax_params.json", "r") as jf:
    synth_softmax_dict = json.load(jf)


# ---------------------------------------------------------------------------
# Pretty print
# ---------------------------------------------------------------------------

def _print_in_columns(items, cols=4, indent="  "):
    items = sorted(items)
    if not items:
        print()
        return

    width = max(len(str(x)) for x in items) + 4
    for i in range(0, len(items), cols):
        row = items[i:i + cols]
        print(indent + "".join(str(x).ljust(width) for x in row).rstrip())
    print()


def pretty_print_available_models(models_dict):
    print("=" * 80)
    print("AVAILABLE MODELS")
    print("=" * 80)
    _print_in_columns(models_dict.keys(), cols=6)


def pretty_print_available_runtimes(runtime_dict):
    print("=" * 80)
    print("AVAILABLE RUNTIMES")
    print("=" * 80)
    _print_in_columns(runtime_dict.keys(), cols=6)


def pretty_print_available_synth_params(synth_dict):
    print("=" * 80)
    print("AVAILABLE SYNTH PARAMS (MUL_MAT)")
    print("=" * 80)

    for key in ["type_a", "m", "n", "k"]:
        values = synth_dict.get(key, [])
        print(f"{key} ({len(values)}):")
        _print_in_columns(values, cols=6)
        
    total = (
        len(synth_dict.get("type_a", []))
        * len(synth_dict.get("type_b", []))
        * len(synth_dict.get("m", []))
        * len(synth_dict.get("n", []))
        * len(synth_dict.get("k", []))
    )
    print(f"Total synth permutations: {total}")
    print()


def pretty_print_available_synth_softmax_params(synth_softmax_dict):
    print("=" * 80)
    print("AVAILABLE SYNTH PARAMS (SOFT_MAX)")
    print("=" * 80)

    for key in ["ne0", "ne1", "ne2", "ne3", "scale", "max_bias"]:
        values = synth_softmax_dict.get(key, [])
        print(f"{key} ({len(values)}):")
        _print_in_columns(values, cols=6)

    mask_variants = synth_softmax_dict.get("mask_variants", [])
    print(f"mask_variants ({len(mask_variants)}):")
    _print_in_columns([json.dumps(v) for v in mask_variants], cols=3)

    sinks = synth_softmax_dict.get("sinks", [])
    print(f"sinks ({len(sinks)}):")
    _print_in_columns(sinks, cols=6)

    total = (
        len(synth_softmax_dict.get("ne0", []))
        * len(synth_softmax_dict.get("ne1", []))
        * len(synth_softmax_dict.get("ne2", []))
        * len(synth_softmax_dict.get("ne3", []))
        * len(mask_variants)
        * len(sinks)
        * len(synth_softmax_dict.get("scale", []))
        * len(synth_softmax_dict.get("max_bias", []))
    )
    print(f"Total synth permutations (before invalid-combo skipping): {total}")
    print()

pretty_print_available_models(models_dict)
pretty_print_available_runtimes(runtime_dict)
pretty_print_available_synth_params(synth_dict)
pretty_print_available_synth_softmax_params(synth_softmax_dict)


AVAILABLE MODELS
  GPT2Q2             Granite4M350Q2     Granite4M350Q3L    Granite4M350Q3M    Granite4M350Q3S    Granite4M350Q4M
  Granite4M350Q4S    Granite4M350Q5M    Granite4M350Q5S    Granite4M350Q6     LFM25M230Q3S       Llama2B7Q3S
  Llama3B1Q2         Llama3B1Q3L        Llama3B1Q3M        Llama3B1Q3S        Llama3B1Q4M        Llama3B1Q4S
  Llama3B1Q5M        Llama3B1Q5S        Llama3B1Q6         Mamba130Q3S        Mamba790Q2         Mamba790Q3L
  Mamba790Q3M        Mamba790Q3S        Mamba790Q4M        Mamba790Q4S        Mamba790Q5M        Mamba790Q5S
  Mamba790Q6         MobileLLMQ2        MobileLLaMAQ2      NanoMistralQ3S     TinyLlama1Q2       TinyLlama2Q2
  TinyLlama2Q3L      TinyLlama2Q3M      TinyLlama2Q3S      TinyLlama2Q4M      TinyLlama2Q4S      TinyLlama2Q5M
  TinyLlama2Q5S      TinyLlama2Q6       llama600M

AVAILABLE RUNTIMES
  bfpbp.q3v1.0A      bfps.q3v1.0A       cpuv1.0            q2q3q4q5q6v1.0A    q2q3q4q5q6v2.0A    q2q3q4q5q6v2.0B
  q2q3q4q5q6v3.0A    q2q3q4q5q

### Edit and Generate Configs

In [3]:
synth_params = {
    "type_a": [
        # "GGML_TYPE_Q2_K",
        "GGML_TYPE_Q3_K",
        # "GGML_TYPE_Q4_K",
        # "GGML_TYPE_Q5_K",
        # "GGML_TYPE_Q6_K",
    ],
    "type_b": ["f32"],
    # "m": [1024, 2048],
    # "n": [1, 16, 32],
    # "k": [1024, 2048, 4096],
    "m": [2048],
    "n": [16],
    "k": [4096],
}

# SOFT_MAX (BFPP_ACC_V4 hardware softmax) synth params. ne0 sweeps across the
# accelerator's SOFTMAX_TILE_N=256 tile boundary (single-tile, exact-tile,
# tile+remainder, exact-2-tiles) to see how per-tile DMA/streaming overhead
# scales; ne2=32/scale=0.125 match TinyLlama-1.1B's n_head=32, head_dim=64
# (scale = 1/sqrt(head_dim)). mask=True/F32 is the realistic causal-attention
# case; max_bias=0.0 since this model doesn't use ALiBi.
synth_softmax_params = {
    "ne0": [128, 256, 300, 512],
    # "ne0": [32, 128, 255, 256, 257, 512, 1024, 2048, 4096],
    "ne1": [1],
    # "ne1": [1, 32],
    "ne2": [32],
    "ne3": [1],
    "mask_variants": [
        {"mask": True, "mask_type": "f32"},
        # {"mask": False},
        # {"mask": True, "mask_type": "f16"},
    ],
    "sinks": [False],
    # "sinks": [False, True],
    "scale": [0.125],
    "max_bias": [0.0],
    # "max_bias": [0.0, 8.0],
}

models = [
    # "MobileLLMQ2",
    # "GPT2Q2",
    # "NanoMistralQ3S",
    # "llama600M",
    "TinyLlama1Q2",
    # "MobileLLaMAQ2",
    # "Mamba790Q3S",
    # "Mamba130Q3S",
    # "Llama2B7Q3S",
]

# runtimes = [
    # "q2v6.0A",
    # "q3v6.0A",
    # "q4v6.0A",
    # "q5v6.0A",
    # "q6v6.0A",
    # "q2q3v6.0A",
    # "q2q4v6.0A",
    # "q2q5v6.0A",
    # "q2q6v6.0A",
    # "q3q4v6.0A",
    # "q3q5v6.0A",
    # "q3q6v6.0A",
    # "q4q5v6.0A",
    # "q4q6v6.0A",
    # "q5q6v6.0A",
    # "q2q3q4v6.0A",
    # "q2q3q5v6.0A",
    # "q2q3q6v6.0A",
    # "q2q4q5v6.0A",
    # "q2q4q6v6.0A",
    # "q2q5q6v6.0A",
    # "q3q4q5v6.0A",
    # "q3q4q6v6.0A",
    # "q3q5q6v6.0A",
    # "q4q5q6v6.0A",
    # "q2q3q4q5v6.0A",
    # "q2q3q4q6v6.0A",
    # "q2q3q5q6v6.0A",
    # "q2q4q5q6v6.0A",
    # "q3q4q5q6v6.0A",
    # "q2q3q4q5q6v6.0A",
    # "cpuv1.0"
# ]

# runtimes = ["cpuv1.0"]


# runtimes = ["q2q3q4q5q6v6.0A" , "q2q3q4q5q6v6.0B", "cpuv1.0"]
# runtimes = ["q2q3q4q5q6v6.0A"]

# runtimes = ["q3v2.0A", "cpuv1.0"]
runtimes = ["bfpbp.q3v1.0A", "cpuv1.0"]
# runtimes = ["cpuv1.0"]

# runtimes = ["cpuv1.0"]




# This loads preconfigured set of runtime x model combinations from a JSON file
# models, runtimes = load_exp_config("configs/exp_configs/Mamba790Q2.json")


# This generates the experiment config file (exp_configs.sh) based on the selected models, runtimes, and synth_params.
# generate_config(runtime_dict, models_dict, runtimes, models, synth_params=synth_params, synth_softmax_params=synth_softmax_params)
generate_config(runtime_dict, models_dict, runtimes, models, synth_params=None, synth_softmax_params=None)
